# Customer Churn Analysis — Telco

## Objective
Identify the key factors influencing customer churn and provide actionable business insights
to improve customer retention.


In [12]:
import pandas as pd
pd.set_option("display.max_columns", None)


## 1. Load dataset

In [13]:
df = pd.read_csv("Telco Customer Churn.csv")
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [14]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 2. Data Cleaning

`TotalCharges` is stored as text because 11 rows contain a blank string instead of a number —
these are new customers with `tenure = 0` who haven't been billed yet, so filling with 0 is the
correct value, not a placeholder for missing data.


In [15]:
print("Blank TotalCharges rows:", (df['TotalCharges'] == ' ').sum())
df[df['TotalCharges'] == ' '][['customerID', 'tenure', 'TotalCharges']]


Blank TotalCharges rows: 11


,customerID,tenure,TotalCharges
488,4472-LVYGI,0,
753,3115-CZMZD,0,
936,5709-LVOEQ,0,
1082,4367-NUYAO,0,
1340,1371-DWPAZ,0,
3331,7644-OMVMY,0,
3826,3213-VVOLG,0,
4380,2520-SGTTA,0,
5218,2923-ARZLG,0,
6670,4075-WKNIU,0,


In [16]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['TotalCharges'].dtype


dtype('float64')

## 3. Validation checks

In [17]:
print("Nulls remaining:\n", df.isnull().sum().sum(), "total")
print()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customerIDs:", df['customerID'].duplicated().sum())
print("customerID count matches row count:", df['customerID'].nunique() == len(df))


Nulls remaining:
 0 total

Duplicate rows: 0
Duplicate customerIDs: 0
customerID count matches row count: True


In [18]:
assert df['tenure'].between(0, 100).all(), "tenure has implausible values"
assert (df['MonthlyCharges'] > 0).all(), "MonthlyCharges has non-positive values"
assert (df['TotalCharges'] >= 0).all(), "TotalCharges has negative values"

print("tenure range:", df['tenure'].min(), "-", df['tenure'].max())
print("MonthlyCharges range:", df['MonthlyCharges'].min(), "-", df['MonthlyCharges'].max())
print("All range checks passed.")


tenure range: 0 - 72
MonthlyCharges range: 18.25 - 118.75
All range checks passed.


**Cleaning summary:** 11 blank `TotalCharges` values filled with 0 (new customers, `tenure = 0`).
No duplicate rows or customer IDs. All value ranges plausible. Dataset is clean and ready for EDA.


## 4. Exploratory Data Analysis

### Overall churn rate

In [19]:
df['Churn'].value_counts()


,count
Churn,
No,5174
Yes,1869


In [20]:
df['Churn'].value_counts(normalize=True).mul(100).round(2)


,proportion
Churn,
No,73.46
Yes,26.54


**Insight:** 26.5% of customers have churned, 73.5% retained — moderate class imbalance,
consistent with this being a real business concern worth deeper analysis.


### Contract type vs churn

In [21]:
pd.crosstab(df['Contract'], df['Churn'], normalize='index').mul(100).round(2)


Churn,No,Yes
Contract,,
Month-to-month,57.29,42.71
One year,88.73,11.27
Two year,97.17,2.83


**Insight:** month-to-month customers churn at 42.7%, vs. 11.3% for one-year and 2.8% for
two-year contracts — contract length is one of the strongest churn predictors in this dataset.


### Tenure vs churn

Bucketed into 0-12 / 12-24 / 24-48 / 48+ months. Uses `-1` as the lower bin edge (not `0`) so that
customers with `tenure = 0` are correctly included in the first bucket — `pd.cut`'s default
interval boundaries are left-exclusive, so a lower edge of `0` would silently exclude every
`tenure = 0` customer from every bucket.


In [22]:
df['tenure'].eq(0).sum()  # customers with tenure = 0, must be included in the 0-12 bucket below


np.int64(11)

In [23]:
tenure_bins = pd.cut(
    df['tenure'],
    bins=[-1, 12, 24, 48, 72],
    labels=['0-12 Months', '12-24 Months', '24-48 Months', '48+ Months']
)

assert tenure_bins.notna().all(), "Some tenure values fell outside all bins — check bin edges"

pd.crosstab(tenure_bins, df['Churn'], normalize='index').mul(100).round(2)


Churn,No,Yes
tenure,,
0-12 Months,52.56,47.44
12-24 Months,71.29,28.71
24-48 Months,79.61,20.39
48+ Months,90.49,9.51


**Insight:** churn rate drops sharply with tenure — 47.4% in the first 12 months, down to 9.5%
for customers past 48 months. Early-stage retention is the single highest-leverage window.


### Tech support vs churn

In [24]:
pd.crosstab(df['TechSupport'], df['Churn'], normalize='index').mul(100).round(2)


Churn,No,Yes
TechSupport,,
No,58.36,41.64
No internet service,92.60,7.40
Yes,84.83,15.17


**Insight:** customers without tech support churn at 41.6%, vs. 15.2% for those who have it —
one of the largest swings of any single feature in this dataset.


### Internet service vs churn

In [25]:
pd.crosstab(df['InternetService'], df['Churn'], normalize='index').mul(100).round(2)


Churn,No,Yes
InternetService,,
DSL,81.04,18.96
Fiber optic,58.11,41.89
No,92.60,7.40


**Insight:** fiber optic customers churn at 41.9%, more than double DSL (19.0%). Customers with
no internet service churn least (7.4%) — likely reflects a different, less price-sensitive segment.


### Online security vs churn

In [26]:
pd.crosstab(df['OnlineSecurity'], df['Churn'], normalize='index').mul(100).round(2)


Churn,No,Yes
OnlineSecurity,,
No,58.23,41.77
No internet service,92.60,7.40
Yes,85.39,14.61


**Insight:** no online security correlates with 41.8% churn vs. 14.6% for subscribers — value-added
services are strongly associated with retention.


### Monthly charges vs churn

In [27]:
df.groupby('Churn')['MonthlyCharges'].mean().round(2)


,MonthlyCharges
Churn,
No,61.27
Yes,74.44


**Insight:** churned customers pay $74.44/month on average vs. $61.27 for retained customers —
higher pricing is associated with higher churn risk.


### Payment method vs churn

In [28]:
pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index').mul(100).round(2)


Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),83.29,16.71
Credit card (automatic),84.76,15.24
Electronic check,54.71,45.29
Mailed check,80.89,19.11


**Insight:** electronic check users churn at 45.3% — by far the highest of any payment method.
Automatic payment methods (bank transfer, credit card) both churn under 17%.


## 5. Export cleaned dataset

In [30]:
df.to_csv("telco churn cleaned.csv", index=False)
print("Exported: telco churn cleaned.csv")
print(f"Final shape: {df.shape}")


Exported: telco churn cleaned.csv
Final shape: (7043, 21)


## 6. Conclusion

Key churn drivers identified: contract type, tenure, tech support availability, internet service
type, online security, monthly charges, and payment method. Early-stage (0-12 month) and
month-to-month customers represent the highest-risk segment. Clean dataset exported for SQL
validation and dashboard development.
